# 13 · 분산 앵커 작업공간 통합 + 검증

ChArUco 보드가 하던 '평면·좌표계·스케일'을 **넓은 작업공간(600×1000mm)에 흩뿌린 마커 지도**로 대체하고, 실제 물체(립밤) 사진으로 검증한다.

- **마커 지도**(`output/marker_map.json`): 19장 다중뷰로 구축, id0 기준 각 마커 위치(코너 일관성 0.35mm).
- **로컬라이제이션**: 사진에 마커 몇 개만 보여도 지도의 알려진 3D 위치로 `solvePnP` → 카메라 포즈.
- 이 포즈 + 작업공간 범위(`plane_xyxy`)를 DA 파이프라인(`process_frame_da`)에 넘기면, **넓은 작업공간 어디든** 물체 검출·측정·3D 배치가 그대로 동작.
- **검증**: 립밤을 보드 물리 중심에 두고 3장(전체보드 1 + 근접 2) 촬영. 기대 위치 ~(0, 430)mm(id0 기준), 크기 H69/D19mm.

In [ ]:
%load_ext autoreload
%autoreload 2
import os, sys, glob
import cv2, numpy as np
import matplotlib.pyplot as plt
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, os.path.join(ROOT, 'src'))
import aruco_utils as au, workspace as ws, scene3d as s3, depth_volume as dv, live_da as ld

OUTPUT_DIR = os.path.join(ROOT, 'output')
CALIB_DIR  = os.path.join(ROOT, 'data', 'calib_images', '31_marker')   # 지도 구축용 19장
SCENE_DIR  = os.path.join(ROOT, 'data', 'scene_images', '31_marker')   # 물체 올린 3장

corners_map, markers_list, REF, L = ws.load_marker_map(os.path.join(OUTPUT_DIR, 'marker_map.json'))
x0, y0, x1, y1 = ws.map_extent_m(corners_map)   # 작업공간 범위(m)
PLANE = (x0, y0, x1, y1)
print(f'마커 {len(corners_map)}개, 작업공간 {(x1-x0)*1000:.0f} x {(y1-y0)*1000:.0f} mm (id{REF} 기준)')
detector = ws.make_detector()

## 0) 카메라 캘리브레이션 (마커 지도 = 캘리브 타겟)

지도를 만든 19장이 곧 캘리브레이션 타겟(각 마커 3D 위치를 앎). 실제 K를 뽑아 저장한다.

> 근사 K(hfov 60)와 fx가 ~1%만 달라 초점거리는 이미 훌륭. 실제 K는 주점(cx,cy) 보정에 의미.

In [ ]:
INTR = os.path.join(OUTPUT_DIR, 'phone_intrinsics.npz')
res = ws.calibrate_from_map(CALIB_DIR, corners_map, detector)
if res is not None:
    K, dist, rms, nviews = res
    np.savez(INTR, K=K, dist=dist, rms=rms)
    print(f'캘리브 {nviews}뷰, RMS {rms:.1f}px  fx={K[0,0]:.0f} fy={K[1,1]:.0f} cx={K[0,2]:.0f} cy={K[1,2]:.0f}')
else:
    print('캘리브 실패 → 근사 K 사용')
    K = dist = None

## 1) 로컬라이제이션 (사진 → 작업공간 좌표계)

물체 사진 한 장으로 데모. 재투영 오차가 낮으면 지도가 실제 마커에 잘 맞은 것.

In [ ]:
f = sorted(glob.glob(os.path.join(SCENE_DIR, '*.jpg')))[-1]   # 근접샷
img = cv2.imread(f); H, W = img.shape[:2]
Kf, distf = (K, dist) if K is not None else au.approx_camera_matrix((W, H), hfov_deg=60)
rvec, tvec, n_used, err = ws.localize(img, corners_map, Kf, distf, detector)
print(f'{os.path.basename(f)}: 사용 마커 {n_used}개, 재투영 오차 {err:.1f}px')
vis = img.copy()
for mid, c in corners_map.items():
    m3 = np.column_stack([c/1000.0, np.zeros(4)])
    p = cv2.projectPoints(m3, rvec, tvec, Kf, distf)[0].reshape(-1, 2).astype(int)
    cv2.polylines(vis, [p], True, (0, 255, 0), 3)
cv2.drawFrameAxes(vis, Kf, distf, rvec, tvec, 0.1, 5)
plt.figure(figsize=(9, 12)); plt.imshow(cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)); plt.axis('off')
plt.title(f'localize: {n_used} markers, reproj {err:.1f}px'); plt.show()

## 2) 작업공간 마커 지도를 3D로 (인터랙티브)

In [ ]:
def shift_markers(items, dx, dy):   # 렌더 원점을 0~로 이동(상대배치 동일)
    out = []
    for m in items:
        c = np.array(m['corners_mm']) + [dx, dy]
        out.append({**m, 'corners_mm': c, 'center_mm': tuple(c.mean(0))})
    return out
DX, DY = -x0*1000, -y0*1000
Wmm, Hmm = (x1-x0)*1000+60, (y1-y0)*1000+60
mk_shift = shift_markers(markers_list, DX, DY)
scene = s3.render_virtual_scene([], markers=mk_shift, ws=(Wmm, Hmm), title='workspace (31 anchors)')
plt.figure(figsize=(8, 11)); plt.imshow(cv2.cvtColor(scene, cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()
s3.render_plotly([], markers=mk_shift, ws=(Wmm, Hmm),
                 html_path=os.path.join(OUTPUT_DIR, 'workspace_map.html'), open_browser=True)
print('인터랙티브: output/workspace_map.html')

## 3) 작업공간 위 물체 검출·측정 (검증)

로컬라이즈 → 포즈·범위·지도를 `process_frame_da`에 전달 → 넓은 작업공간에서 DA 검출·측정·3D 배치.
3장 모두 실행: 근접샷(전체 보드 안 보임)에서도 물체 위치·크기가 나오는지 확인.

In [ ]:
pipe = dv.load_depth_model()

def detect_on_workspace(frame, Kf, distf, pipe, show=True):
    r = ws.localize(frame, corners_map, Kf, distf, detector)
    if r is None:
        print('로컬라이즈 실패(마커 부족)'); return None
    rvec, tvec, n, err = r
    vis, objs, _ = ld.process_frame_da(frame, pipe, None, Kf, distf,
                                       pose=(rvec, tvec), plane_xyxy=PLANE,
                                       marker_map=markers_list, min_area_px=1200)
    print(f'  로컬라이즈 {n}마커({err:.1f}px), 물체 {len(objs)}개')
    for o in objs:
        c = o['cyl'][0]
        print(f"    {o['type']:5s} 위치=({c[0]:+.0f},{c[1]:+.0f})mm  {o['label']}")
    return vis, objs

print('기대: 위치~(0,430)mm, 크기 H69/D19')
files = sorted(glob.glob(os.path.join(SCENE_DIR, '*.jpg')))
fig, axes = plt.subplots(1, len(files), figsize=(5*len(files), 7))
for ax, f in zip(np.atleast_1d(axes), files):
    im = cv2.imread(f); Hh, Ww = im.shape[:2]
    Kf, distf = (K, dist) if K is not None else au.approx_camera_matrix((Ww, Hh), hfov_deg=60)
    print(os.path.basename(f))
    out = detect_on_workspace(im, Kf, distf, pipe)
    if out:
        ax.imshow(cv2.cvtColor(cv2.resize(out[0], (Ww//4, Hh//4)), cv2.COLOR_BGR2RGB))
    ax.set_title(os.path.basename(f)); ax.axis('off')
plt.tight_layout(); plt.show()

## 정리 & 결과

- **마커 지도 → 로컬라이제이션 → DA 파이프라인** 통합·검증 완료. 보드 없이 넓은 작업공간에서 동작.
- **검증 결과(근접샷 2장)**: 립밤 위치 ~(+6, +440)mm ≈ 기대 (0, 430), 크기 H~70/D~16 ≈ 실제 69/19. **전체 보드가 안 보이는 근접샷에서도** 주변 마커만으로 3D 위치·크기 복원 성공.
- **한계**: 전체 보드를 멀리서 담은 원거리 샷은 물체(19mm)가 DA 해상 한계로 검출 안 됨 → 가까이 촬영 필요. 넓고 비스듬한 뷰는 DA 상대깊이가 먼 쪽으로 드리프트(타당성 필터 `max_dim_mm`로 가짜 거대 블롭 제거).
- **로봇팔 적용**: 카메라를 작업공간 위 근접/탑다운으로 고정하면 근접샷 조건과 같아 정확. 
- 다음: 최소 앵커 수 실험(k개만으로 로컬라이즈 오차), 실시간(웹캠) 작업공간 연동.